# 5 datasets - Cold-cell split - Ridge, Lasso, KNN, Random Forest

Notebook này chạy 4 model cổ điển trên 5 dataset từ CSA/IMPROVE: `CCLE`, `CTRPv2`, `gCSI`, `GDSCv1`, `GDSCv2`.

Pipeline:

- target: `auc` trong `response.tsv`
- split: tự tạo **cold split theo cell line** (`improve_sample_id`) với tỉ lệ train/val/test = 70/10/20
- drug feature: ECFP4 512-bit có sẵn trong CSA
- cell-line feature: gene expression, fit `StandardScaler + PCA` trên train cell lines rồi transform val/test
- model: Ridge, Lasso, KNN, Random Forest

Cold split nghĩa là cell line trong validation/test không xuất hiện trong train. Notebook cũng in diagnostics overlap để kiểm tra leakage.

In [ ]:
# Kaggle setup: run this cell first.
!pip install -q numpy pandas scikit-learn matplotlib joblib


In [ ]:
from pathlib import Path
import gc
import json
import os
import time
import warnings

CACHE_DIR = Path('../.cache')
(CACHE_DIR / 'matplotlib').mkdir(parents=True, exist_ok=True)
(CACHE_DIR / 'fontconfig').mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str((CACHE_DIR / 'matplotlib').resolve()))
os.environ.setdefault('XDG_CACHE_HOME', str(CACHE_DIR.resolve()))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Lasso, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 160)

## 1. Config

In [ ]:
RANDOM_STATE = 42
PCA_N_COMPONENTS = 0.95

SPLIT_METHOD = 'tdc_style_cold_split'
GROUP_COL = 'cell_id'
SPLIT_FRAC = [0.7, 0.1, 0.2]  # same idea as TDC: frac=[0.7, 0.1, 0.2]
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = SPLIT_FRAC

# None = chạy full benchmark. Đặt số nhỏ, ví dụ 5000, nếu muốn smoke test nhanh.
MAX_ROWS_PER_SPLIT = None

DATASETS = ['CCLE', 'CTRPv2', 'gCSI', 'GDSCv1', 'GDSCv2']

DATA_DIR = Path('../data/csa_data/raw_data')
X_DIR = DATA_DIR / 'x_data'
Y_DIR = DATA_DIR / 'y_data'
OUTPUT_DIR = Path('../models/five_dataset_4models_results_cold_cell')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODELS = {
    'ridge': Ridge(alpha=100.0),
    'lasso': Lasso(alpha=0.001, max_iter=5000, tol=1e-3, selection='random', random_state=RANDOM_STATE),
    'knn': KNeighborsRegressor(n_neighbors=31, weights='distance', metric='minkowski', p=2, n_jobs=-1),
    'rf': RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=2,
        max_features='sqrt',
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
}

print('Datasets:', DATASETS)
print('Split:', SPLIT_METHOD, f'frac={SPLIT_FRAC}', 'by', GROUP_COL)
print('Output:', OUTPUT_DIR.resolve())

## 2. Load Raw Tables

In [ ]:
def require_file(path):
    if not Path(path).exists():
        raise FileNotFoundError(f'Missing file: {path}')

required_files = [
    Y_DIR / 'response.tsv',
    X_DIR / 'drug_ecfp4_nbits512.tsv',
    X_DIR / 'cancer_gene_expression.tsv',
]
for path in required_files:
    require_file(path)

response = pd.read_csv(Y_DIR / 'response.tsv', sep='\t', low_memory=False)

# File gene expression có 2 dòng metadata sau header: Entrez ID và gene symbol.
gene_expression = pd.read_csv(
    X_DIR / 'cancer_gene_expression.tsv',
    sep='\t',
    index_col=0,
    skiprows=[1, 2],
).astype(np.float32)

drug_ecfp = pd.read_csv(X_DIR / 'drug_ecfp4_nbits512.tsv', sep='\t')
drug_ecfp = drug_ecfp.set_index('improve_chem_id').astype(np.float32)

print('response:', response.shape)
print('gene_expression:', gene_expression.shape)
print('drug_ecfp:', drug_ecfp.shape)
print('sources:', sorted(response['source'].dropna().unique()))

## 3. Helpers

In [ ]:
def make_base_dataframe(dataset):
    df = response.loc[response['source'] == dataset, ['source', 'improve_sample_id', 'improve_chem_id', 'auc']].copy()
    df = df.rename(columns={
        'improve_sample_id': 'cell_id',
        'improve_chem_id': 'drug_id',
        'auc': 'y',
    })
    df = df.dropna(subset=['cell_id', 'drug_id', 'y'])
    df = df[df['cell_id'].isin(gene_expression.index)]
    df = df[df['drug_id'].isin(drug_ecfp.index)]
    df = df.reset_index(drop=True)
    if df.empty:
        raise ValueError(f'No usable rows for dataset {dataset}')
    return df


def make_tdc_style_cold_split(df):
    # Mirrors the TDC call used in data_eda_pca_preparation.ipynb:
    # data.get_split(method='cold_split', column_name='Cell Line_ID', seed=42, frac=[0.7, 0.1, 0.2])
    # GroupShuffleSplit uses fractions over unique groups, so validation/test cell lines are unseen in train.
    train_splitter = GroupShuffleSplit(
        n_splits=1,
        train_size=SPLIT_FRAC[0],
        random_state=RANDOM_STATE,
    )
    train_idx, holdout_idx = next(train_splitter.split(df, groups=df[GROUP_COL]))

    train_df = df.iloc[train_idx].reset_index(drop=True)
    holdout_df = df.iloc[holdout_idx].reset_index(drop=True)

    valid_share = SPLIT_FRAC[1] / (SPLIT_FRAC[1] + SPLIT_FRAC[2])
    valid_test_splitter = GroupShuffleSplit(
        n_splits=1,
        train_size=valid_share,
        random_state=RANDOM_STATE,
    )
    valid_idx, test_idx = next(valid_test_splitter.split(holdout_df, groups=holdout_df[GROUP_COL]))

    valid_df = holdout_df.iloc[valid_idx].reset_index(drop=True)
    test_df = holdout_df.iloc[test_idx].reset_index(drop=True)
    return train_df, valid_df, test_df


def maybe_limit_rows(df):
    if MAX_ROWS_PER_SPLIT is not None and len(df) > MAX_ROWS_PER_SPLIT:
        return df.sample(MAX_ROWS_PER_SPLIT, random_state=RANDOM_STATE).reset_index(drop=True)
    return df


def split_overlap_report(train_df, val_df, test_df):
    rows = []
    train_cells = set(train_df['cell_id'])
    train_drugs = set(train_df['drug_id'])
    train_pairs = set(zip(train_df['cell_id'], train_df['drug_id']))
    for split_name, split_df in [('val', val_df), ('test', test_df)]:
        split_cells = set(split_df['cell_id'])
        split_drugs = set(split_df['drug_id'])
        split_pairs = set(zip(split_df['cell_id'], split_df['drug_id']))
        rows.append({
            'split': split_name,
            'rows': int(len(split_df)),
            'unique_cells': int(split_df['cell_id'].nunique()),
            'unique_drugs': int(split_df['drug_id'].nunique()),
            'cell_overlap_with_train': int(len(train_cells & split_cells)),
            'drug_overlap_with_train': int(len(train_drugs & split_drugs)),
            'pair_overlap_with_train': int(len(train_pairs & split_pairs)),
        })
    report = pd.DataFrame(rows)
    assert (report['cell_overlap_with_train'] == 0).all(), 'Cold-cell split failed: cell overlap with train found.'
    return report


def fit_cell_pca(train_df, all_split_df):
    train_cells = train_df['cell_id'].drop_duplicates().tolist()
    all_cells = all_split_df['cell_id'].drop_duplicates().tolist()

    train_matrix = gene_expression.loc[train_cells].to_numpy(dtype=np.float32)
    all_matrix = gene_expression.loc[all_cells].to_numpy(dtype=np.float32)

    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_matrix)

    pca = PCA(n_components=PCA_N_COMPONENTS, random_state=RANDOM_STATE)
    pca.fit(train_scaled)

    all_cell_pca = pca.transform(scaler.transform(all_matrix)).astype(np.float32)
    cell_pca_map = dict(zip(all_cells, all_cell_pca))
    return scaler, pca, cell_pca_map


def build_xy(df, cell_pca_map):
    drug_x = drug_ecfp.loc[df['drug_id']].to_numpy(dtype=np.float32)
    cell_x = np.vstack([cell_pca_map[cell_id] for cell_id in df['cell_id']]).astype(np.float32)
    x = np.hstack([drug_x, cell_x]).astype(np.float32)
    y = df['y'].to_numpy(dtype=np.float32)
    return x, y


def prepare_dataset(dataset):
    base_df = make_base_dataframe(dataset)
    train_df, val_df, test_df = make_tdc_style_cold_split(base_df)
    train_df = maybe_limit_rows(train_df)
    val_df = maybe_limit_rows(val_df)
    test_df = maybe_limit_rows(test_df)
    all_split_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

    overlap_report = split_overlap_report(train_df, val_df, test_df)
    display(overlap_report)

    scaler, pca, cell_pca_map = fit_cell_pca(train_df, all_split_df)

    x_train, y_train = build_xy(train_df, cell_pca_map)
    x_val, y_val = build_xy(val_df, cell_pca_map)
    x_test, y_test = build_xy(test_df, cell_pca_map)

    metadata = {
        'dataset': dataset,
        'split_method': SPLIT_METHOD,
        'group_column': GROUP_COL,
        'split_frac': SPLIT_FRAC,
        'train_frac': TRAIN_FRAC,
        'valid_frac': VAL_FRAC,
        'test_frac': TEST_FRAC,
        'random_state': RANDOM_STATE,
        'all_usable_rows': int(len(base_df)),
        'train_rows': int(len(train_df)),
        'valid_rows': int(len(val_df)),
        'test_rows': int(len(test_df)),
        'train_cells': int(train_df['cell_id'].nunique()),
        'valid_cells': int(val_df['cell_id'].nunique()),
        'test_cells': int(test_df['cell_id'].nunique()),
        'n_drugs': int(all_split_df['drug_id'].nunique()),
        'drug_feature': 'CSA ECFP4 512-bit',
        'cell_feature': 'StandardScaler + PCA fitted on train cell lines only',
        'pca_components': int(pca.n_components_),
        'pca_explained_variance': float(np.sum(pca.explained_variance_ratio_)),
        'x_train_shape': list(x_train.shape),
        'x_val_shape': list(x_val.shape),
        'x_test_shape': list(x_test.shape),
        'max_rows_per_split': MAX_ROWS_PER_SPLIT,
    }

    dataset_dir = OUTPUT_DIR / dataset
    dataset_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(scaler, dataset_dir / 'cell_line_scaler.joblib')
    joblib.dump(pca, dataset_dir / 'cell_line_pca.joblib')
    overlap_report.to_csv(dataset_dir / 'split_overlap_report.csv', index=False)
    with open(dataset_dir / 'metadata.json', 'w') as f:
        json.dump(metadata, f, indent=2)

    print(f"{dataset}: train={x_train.shape}, val={x_val.shape}, test={x_test.shape}, PCA={pca.n_components_} comps")
    return metadata, (x_train, y_train, x_val, y_val, x_test, y_test)

## 4. Train / Validation / Test Metrics

In [ ]:
def safe_pearson(y_true, y_pred):
    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        return np.nan
    return float(np.corrcoef(y_true, y_pred)[0, 1])


def regression_metrics(y_true, y_pred):
    return {
        'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'r2': float(r2_score(y_true, y_pred)),
        'pearson': safe_pearson(y_true, y_pred),
    }


def fit_and_eval(dataset, model_name, base_model, arrays):
    x_train, y_train, x_val, y_val, x_test, y_test = arrays
    dataset_dir = OUTPUT_DIR / dataset

    model = clone(base_model)
    start = time.time()
    model.fit(x_train, y_train)
    fit_seconds = time.time() - start

    pred_train = model.predict(x_train)
    pred_val = model.predict(x_val)
    pred_test = model.predict(x_test)

    row = {
        'dataset': dataset,
        'model': model_name,
        'train_size': int(len(y_train)),
        'val_size': int(len(y_val)),
        'test_size': int(len(y_test)),
        'fit_seconds': float(fit_seconds),
        **{f'train_{k}': v for k, v in regression_metrics(y_train, pred_train).items()},
        **{f'val_{k}': v for k, v in regression_metrics(y_val, pred_val).items()},
        **{f'test_{k}': v for k, v in regression_metrics(y_test, pred_test).items()},
    }

    joblib.dump(model, dataset_dir / f'{model_name}.joblib')
    np.save(dataset_dir / f'{model_name}_val_pred.npy', pred_val)
    np.save(dataset_dir / f'{model_name}_test_pred.npy', pred_test)
    return row

## 5. Run All 5 Datasets

In [ ]:
all_metadata = []
all_results = []

for dataset in DATASETS:
    print('\n' + '=' * 90)
    print('Dataset:', dataset)
    metadata, arrays = prepare_dataset(dataset)
    all_metadata.append(metadata)

    dataset_rows = []
    for model_name, model in MODELS.items():
        print('Training:', model_name)
        row = fit_and_eval(dataset, model_name, model, arrays)
        dataset_rows.append(row)
        print(f"  val_rmse={row['val_rmse']:.4f} | test_rmse={row['test_rmse']:.4f} | fit={row['fit_seconds']:.1f}s")

    dataset_results = pd.DataFrame(dataset_rows).sort_values('val_rmse')
    dataset_results.to_csv(OUTPUT_DIR / dataset / 'results.csv', index=False)
    all_results.append(dataset_results)

    display(dataset_results.round(4))

    del arrays, dataset_rows, dataset_results
    gc.collect()

metadata_df = pd.DataFrame(all_metadata)
results_df = pd.concat(all_results, ignore_index=True).sort_values(['dataset', 'val_rmse'])

metadata_df.to_csv(OUTPUT_DIR / 'metadata_all_datasets.csv', index=False)
results_df.to_csv(OUTPUT_DIR / 'results_all_datasets.csv', index=False)

print('Saved:', (OUTPUT_DIR / 'results_all_datasets.csv').resolve())
display(metadata_df)
display(results_df.round(4))

## 6. Best Model and Plots

In [ ]:
best_by_dataset = results_df.loc[results_df.groupby('dataset')['val_rmse'].idxmin()].sort_values('val_rmse')
best_by_dataset.to_csv(OUTPUT_DIR / 'best_model_by_dataset.csv', index=False)
display(best_by_dataset.round(4))

plot_df = results_df.copy()
model_order = list(MODELS.keys())
dataset_order = DATASETS
x = np.arange(len(dataset_order))
width = 0.18

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharex=True)
for i, model_name in enumerate(model_order):
    subset = plot_df[plot_df['model'] == model_name].set_index('dataset').reindex(dataset_order)
    offset = (i - (len(model_order) - 1) / 2) * width
    axes[0].bar(x + offset, subset['val_rmse'], width=width, label=model_name)
    axes[1].bar(x + offset, subset['test_rmse'], width=width, label=model_name)

axes[0].set_title('Validation RMSE')
axes[1].set_title('Test RMSE')
for ax in axes:
    ax.set_xticks(x)
    ax.set_xticklabels(dataset_order, rotation=25, ha='right')
    ax.set_ylabel('RMSE')
    ax.grid(axis='y', alpha=0.25)

axes[1].legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'rmse_by_dataset_model.png', dpi=180, bbox_inches='tight')
plt.show()

print('All outputs in:', OUTPUT_DIR.resolve())